In [21]:
## 1. Test LLM connection

from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:18000/v1",
    api_key="dummy",
)

MODEL_NAME = "Qwen/Qwen3.5-397B-A17B-FP8"


response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Return only JSON: {\"label\":\"negative\"}"
        }
    ],
)

print(response.choices[0].message.content)



{"label":"negative"}


In [26]:
## 2. Load compact LLM review dataset

import json
from pathlib import Path

LLM_INPUT_PATH = Path(
    "../data/processed/porseman_llm_review_compact.jsonl"
)

with LLM_INPUT_PATH.open(
    encoding="utf-8"
) as f:
    first_record = json.loads(next(f))

print("ID:")
print(first_record["id"])

print("\nQuestion:")
print(first_record["query"])

print("\nPositive:")
print(first_record["positive"][:500])

print("\nFirst candidate:")
print(first_record["candidates"][0])

ID:
porseman-12457

Question:
روزه‌هایى که در اوایل سن تکلیف به جا نیاورده‌ام، علاوه بر قضا کفّاره هم دارد؟

Positive:
همه مراجع: هر مقدار از روزه‌ها را که نگرفته‌اید، باید قضا کنید و افزون بر آن، براى هر روز نیز باید کفّاره بدهید؛ یعنى، دو ماه روزه بگیرید یا شصت فقیر را سیر کنید و یا به هر کدام یک مد (تقریبا ده سیر) طعام (گندم یا جو و مانند آن) به آنها بدهید. [ توضیح‌المسائل مراجع، م 1660؛ وحید، توضیح‌المسائل، م 1668.]

First candidate:
{'candidate_id': 'porseman-34656', 'retrieval_rank': 13, 'retrieval_score': 0.68994140625, 'reranker_score': 0.9742394685745239}


In [27]:
## 3. Build answer lookup from the same JSONL

import json

records_by_id = {}

with LLM_INPUT_PATH.open(
    encoding="utf-8"
) as f:
    for line in f:
        record = json.loads(line)
        records_by_id[record["id"]] = record


print(f"Loaded records: {len(records_by_id):,}")

candidate_id = first_record["candidates"][0]["candidate_id"]

candidate_record = records_by_id[candidate_id]

print("\nCandidate ID:")
print(candidate_id)

print("\nCandidate question:")
print(candidate_record["query"])

print("\nCandidate answer:")
print(candidate_record["positive"][:500])

Loaded records: 12,807

Candidate ID:
porseman-34656

Candidate question:
سوال من در مورد قضای روزه های زمانی است که تازه به تکلیف رسیده بودم. من از سن ۹ سالگی که به تکلیف رسیدم تا تقریبا سن ۱۳ سالگی روزه هایم را از روی جهالت نمی گرفتم و مادرم هم چون جثه کوچکی داشتم به من چیزی نمی گفتند یعنی هر وقت می خواستم می گرفتم و هر وقت نمی خواستم نمی گرفتم می خواستم بپرسم الان تکلیفم چیست و آیا علاوه بر قضاهای آن روزه ها باید کفاره هم بدهم یا نه؟ کفاره اش چیست؟ ضمنا گاهی اوقات هم روزه می گرفتم اما گرسنه ام که می شد یا دوستم چیزی تعارف می کرد می خوردم حکم آنها چیست؟ البته در تمام این موارد نمی دانستم که حکم چیست؟

Candidate answer:
تمام روزه هایی را که یقین دارید بعد از رسیدن به سن بلوغ بدون عذر شرعی نگرفته یا باطل کرده اید باید قضا نمایید و اگر در ان حال توجه به مسئله داشته و می دانسته اید که نباید روزه را باطل کنید علاوه بر قضا باید برای هر روز کفاره هم بدهید و کفاره یکی از دو چیز است - الف - برای هرروز 60 روز روزه گرفتن که 31 روزش پی در پی باشد - ب - برای هرروز 60 فقیر را طعام دادن یعنی به هر 

In [28]:
## 4. Test one real candidate annotation

import json

question = first_record["query"]
positive = first_record["positive"]
candidate = candidate_record["positive"]

prompt = f"""
You are a dataset annotator for training an embedding model.

Compare the candidate answer with the correct answer.

Question:
{question}

Correct answer:
{positive}

Candidate answer:
{candidate}

Classify the candidate as exactly one of:

- negative: the candidate is incorrect, contradicts the correct answer, or does not answer the question.
- equivalent: the candidate gives the same correct answer or the same meaning as the correct answer.
- uncertain: you cannot confidently decide.

Return only valid JSON:
{{"label":"negative"}}
"""

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
)

output = response.choices[0].message.content

print(output)



{"label":"equivalent"}


In [32]:
## 5. Test one real candidate annotation (thinking disabled)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    temperature=0,
    max_tokens=20,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    },
)

print(response.choices[0].message.content)

{
  "label": "negative"
}


In [33]:
## 6. Test LLM labeling on 10 candidates

import json

VALID_LABELS = {
    "negative",
    "equivalent",
    "uncertain",
}

test_results = []

record = first_record

for candidate_meta in record["candidates"][:10]:
    candidate_id = candidate_meta["candidate_id"]
    candidate_text = records_by_id[candidate_id]["positive"]

    prompt = f"""
You are a dataset annotator for training an embedding model.

Compare the candidate answer with the correct answer for the given question.

Question:
{record["query"]}

Correct answer:
{record["positive"]}

Candidate answer:
{candidate_text}

Choose exactly one label:

negative:
The candidate is incorrect, contradicts the correct answer, or does not correctly answer the question.

equivalent:
The candidate gives the same correct answer or has the same meaning as the correct answer.

uncertain:
The relationship is ambiguous or you cannot confidently classify it.

Return only a JSON object with exactly one field named "label".
The label must be one of:
negative, equivalent, uncertain.
""".strip()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
        max_tokens=20,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        },
    )

    raw_output = response.choices[0].message.content
    result = json.loads(raw_output)

    label = result["label"]

    assert label in VALID_LABELS

    test_results.append(
        {
            "candidate_id": candidate_id,
            "label": label,
        }
    )

    print(candidate_id, "->", label)

porseman-34656 -> negative
porseman-12610 -> negative
porseman-35427 -> negative
porseman-12542 -> negative
porseman-12782 -> negative
porseman-34834 -> negative
porseman-12678 -> negative
porseman-12473 -> negative
porseman-12499 -> negative
porseman-1047 -> negative


In [34]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    temperature=0,
    max_tokens=20,

    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "candidate_label",
            "schema": {
                "type": "object",
                "properties": {
                    "label": {
                        "type": "string",
                        "enum": [
                            "negative",
                            "equivalent",
                            "uncertain"
                        ]
                    }
                },
                "required": ["label"],
                "additionalProperties": False
            }
        }
    },

    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    },
)

print(response.choices[0].message.content)

{
  "label": "negative"
}


In [35]:
## 7. Test labeling across 10 different questions

import json
from collections import Counter
from tqdm.auto import tqdm

VALID_LABELS = {
    "negative",
    "equivalent",
    "uncertain",
}

sample_records = list(records_by_id.values())[:10]

test_results = []

for record in tqdm(
    sample_records,
    desc="LLM labeling",
):
    candidate_meta = record["candidates"][0]

    candidate_id = candidate_meta["candidate_id"]
    candidate_text = records_by_id[candidate_id]["positive"]

    prompt = f"""
You are a dataset annotator for training an embedding model.

Compare the candidate answer with the correct answer for the given question.

Question:
{record["query"]}

Correct answer:
{record["positive"]}

Candidate answer:
{candidate_text}

Choose exactly one label:

negative:
The candidate is incorrect, contradicts the correct answer, or does not correctly answer the question.

equivalent:
The candidate gives the same correct answer or has the same meaning as the correct answer.

uncertain:
The relationship is ambiguous or you cannot confidently classify it.
""".strip()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
        max_tokens=20,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "candidate_label",
                "schema": {
                    "type": "object",
                    "properties": {
                        "label": {
                            "type": "string",
                            "enum": [
                                "negative",
                                "equivalent",
                                "uncertain",
                            ],
                        }
                    },
                    "required": ["label"],
                    "additionalProperties": False,
                },
            },
        },
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        },
    )

    result = json.loads(
        response.choices[0].message.content
    )

    label = result["label"]

    assert label in VALID_LABELS

    test_results.append(
        {
            "query_id": record["id"],
            "candidate_id": candidate_id,
            "label": label,
        }
    )

print("\nResults:")
for item in test_results:
    print(
        item["query_id"],
        "->",
        item["candidate_id"],
        "->",
        item["label"],
    )

print("\nLabel counts:")
print(Counter(x["label"] for x in test_results))

c:\Users\1\Documents\finetune-embedding-models\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
LLM labeling: 100%|██████████| 10/10 [00:05<00:00,  1.68it/s]


Results:
porseman-12457 -> porseman-34656 -> negative
porseman-40730 -> porseman-22827 -> equivalent
porseman-22797 -> porseman-1895 -> equivalent
porseman-3019 -> porseman-5207 -> equivalent
porseman-6283 -> porseman-26749 -> negative
porseman-36735 -> porseman-35765 -> negative
porseman-44062 -> porseman-18593 -> negative
porseman-13261 -> porseman-45675 -> equivalent
porseman-16434 -> porseman-25277 -> negative
porseman-46696 -> porseman-29246 -> equivalent

Label counts:
Counter({'negative': 5, 'equivalent': 5})


In [36]:
## 8. Test 50 random query-candidate pairs

import random
import json
from collections import Counter
from tqdm.auto import tqdm

random.seed(42)

sample_records = random.sample(
    list(records_by_id.values()),
    50,
)

test_results = []

for record in tqdm(
    sample_records,
    desc="LLM labeling",
):
    candidate_meta = random.choice(record["candidates"])

    candidate_id = candidate_meta["candidate_id"]
    candidate_text = records_by_id[candidate_id]["positive"]

    prompt = f"""
You are a dataset annotator for training an embedding model.

Compare the candidate answer with the correct answer for the given question.

Question:
{record["query"]}

Correct answer:
{record["positive"]}

Candidate answer:
{candidate_text}

Choose exactly one label:

negative:
The candidate is incorrect, contradicts the correct answer, or does not correctly answer the question.

equivalent:
The candidate gives the same correct answer or has the same meaning as the correct answer.

uncertain:
The relationship is ambiguous or you cannot confidently classify it.
""".strip()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
        max_tokens=20,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "candidate_label",
                "schema": {
                    "type": "object",
                    "properties": {
                        "label": {
                            "type": "string",
                            "enum": [
                                "negative",
                                "equivalent",
                                "uncertain",
                            ],
                        }
                    },
                    "required": ["label"],
                    "additionalProperties": False,
                },
            },
        },
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        },
    )

    result = json.loads(
        response.choices[0].message.content
    )

    test_results.append(
        {
            "query_id": record["id"],
            "candidate_id": candidate_id,
            "candidate_position": record["candidates"].index(candidate_meta) + 1,
            "label": result["label"],
        }
    )

print("\nLabel counts:")
print(Counter(x["label"] for x in test_results))

LLM labeling: 100%|██████████| 50/50 [00:28<00:00,  1.77it/s]


Label counts:
Counter({'negative': 43, 'equivalent': 7})


In [37]:
## 9. Inspect equivalent labels

equivalent_results = [
    x for x in test_results
    if x["label"] == "equivalent"
]

for item in equivalent_results:
    query_record = records_by_id[item["query_id"]]
    candidate_record = records_by_id[item["candidate_id"]]

    print("=" * 100)
    print(f"Query ID: {item['query_id']}")
    print(f"Candidate ID: {item['candidate_id']}")
    print(f"Candidate position: {item['candidate_position']}")

    print("\nQUESTION:")
    print(query_record["query"])

    print("\nPOSITIVE:")
    print(query_record["positive"])

    print("\nCANDIDATE:")
    print(candidate_record["positive"])

    print()

Query ID: porseman-40264
Candidate ID: porseman-1703
Candidate position: 4

QUESTION:
آیا قبل از تشریع حجاب حضرت زهرا(س) حجاب داشتند؟

POSITIVE:
پیش از ظهور اسلام ,درمیان زنان شبه جزیره, حجاب متعارف بود و چنین نبود که پس از آمدن اسلام در میان آنان حجاب پیدا شود. زنان عرب به شکلی لباس می پوشیدند که یقه و سینه هایشان باز می ماند. آنان روسری خود را به این صورت به سر می کردند که گردن و زیر گلو هم باز می ماند .آنها دامنه روسری را به پشت سر می انداختند .دراین نوع پوشش بطور طبیعی گوشواره ها , گردنبند ها وگردن و سینه باز می ماند.آیه آمد که دامنه روسری ها را به روی سینه بیندازید تا گردن و سینه پوشیده شود .(1)زنان عربی دو گونه روسری داشتند : یکی روسری کوچک بود که در خانه از آن استفاده می کردند و یکی هم روسری بزرگ بود و ازآن به عنوان تشریفات و تزیین استفاده می کردند.آیه آمد که از روسری بزرگ به عنوان پوشش استفاده کنید و همواره آن را با خود داشته باشید و به این ترتیب حجاب کامل شد.(2) این روسری بزرگ , از چادر کوچکتر و از روسری بزرگتر بود.(3)بر این اساس همه زنان عربی حجاب داشتند و فاطمه زهرا سلام الل

In [38]:
## 10. Test stricter LLM labeling

import json
import random
from collections import Counter
from tqdm.auto import tqdm

STRICT_PROMPT_TEMPLATE = """
You are a strict dataset annotator for training a retrieval embedding model.

Your task is to determine whether the candidate answer is a valid answer to the specific question.

Question:
{question}

Correct answer:
{positive}

Candidate answer:
{candidate}

Choose exactly one label:

negative:
The candidate is incorrect, contradicts the correct answer, does not answer the specific question, or answers a different but related question.
A candidate is still negative even if it discusses the same topic or uses very similar terminology but fails to answer the specific question correctly.

equivalent:
The candidate actually answers the specific question and reaches substantially the same conclusion as the correct answer.
Wording, level of detail, and supporting explanations may differ.
Topic similarity alone is NOT enough for this label.

uncertain:
There is not enough information to confidently determine whether the candidate correctly answers the specific question.

Be strict when assigning equivalent.
""".strip()


sample_records = random.sample(
    list(records_by_id.values()),
    50,
)

test_results = []

for record in tqdm(
    sample_records,
    desc="LLM labeling",
):
    candidate_meta = random.choice(record["candidates"])

    candidate_id = candidate_meta["candidate_id"]
    candidate_text = records_by_id[candidate_id]["positive"]

    prompt = STRICT_PROMPT_TEMPLATE.format(
        question=record["query"],
        positive=record["positive"],
        candidate=candidate_text,
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
        max_tokens=20,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "candidate_label",
                "schema": {
                    "type": "object",
                    "properties": {
                        "label": {
                            "type": "string",
                            "enum": [
                                "negative",
                                "equivalent",
                                "uncertain",
                            ],
                        }
                    },
                    "required": ["label"],
                    "additionalProperties": False,
                },
            },
        },
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        },
    )

    result = json.loads(
        response.choices[0].message.content
    )

    test_results.append(
        {
            "query_id": record["id"],
            "candidate_id": candidate_id,
            "candidate_position": record["candidates"].index(candidate_meta) + 1,
            "label": result["label"],
        }
    )

print("\nLabel counts:")
print(Counter(x["label"] for x in test_results))

LLM labeling: 100%|██████████| 50/50 [00:21<00:00,  2.29it/s]


Label counts:
Counter({'negative': 49, 'equivalent': 1})


In [39]:
## 11. Inspect labels from the stricter prompt

equivalents = [
    x for x in test_results
    if x["label"] == "equivalent"
]

negatives = [
    x for x in test_results
    if x["label"] == "negative"
]

print("=== EQUIVALENT ===")

for item in equivalents:
    record = records_by_id[item["query_id"]]
    candidate = records_by_id[item["candidate_id"]]

    print("\n" + "=" * 100)
    print("QUESTION:")
    print(record["query"])

    print("\nPOSITIVE:")
    print(record["positive"])

    print("\nCANDIDATE:")
    print(candidate["positive"])


print("\n\n=== 5 SAMPLE NEGATIVES ===")

for item in negatives[:5]:
    record = records_by_id[item["query_id"]]
    candidate = records_by_id[item["candidate_id"]]

    print("\n" + "=" * 100)
    print("QUESTION:")
    print(record["query"])

    print("\nPOSITIVE:")
    print(record["positive"])

    print("\nCANDIDATE:")
    print(candidate["positive"])

=== EQUIVALENT ===

QUESTION:
لطفأ حدیثی درمورد توکل کردن بخدا بفرمایید؟

POSITIVE:
در پاسخ به دو روایت ذیل توجه کنید : 1- در حدیثی رسول گرامی اسلام(ص) از جبرئیل(ع) می پرسد: توکل بر خداوند متعال چیست؟ جبرئیل(ع) در پاسخ می گوید: «العلم بان المخلوق لایضر و لا ینفع ولا یعطی و لا یمنع و استعمال الیأس من الخلق؛ حقیقت توکل علم و آگاهی به این است که مخلوق نمی تواند زیانی برساند و نه سودی و نه چیزی ببخشد و نه از آن باز دارد، و نیز توکل مأیوس شدن از خلق است (یعنی همه چیز را از خدا و به فرمان او بداند) (بحار، ج 68، ص 138، ح 23).2- در حدیث دیگری می خوانیم که معصوم(ع) در پاسخ پرسش درباره توکل فرمود: «لا تخاف سواه» توکل این است که از غیر خدا نترسی. (بحار، ج 68، ص 143 ، ح 42).

CANDIDATE:
با سلام به شما پرسشگر گرامی و با سپاس از اینکه مرکز ما را برای پاسخ و راهنمایی برگزیده اید. برای پاسخ به سؤالتان، ابتدا به تعریف «توکل» می‌پردازیم: معنای توکل: توکل از ماده وکالت به معنای سپردن کارها به خداوند و اعتماد بر لطف او است. مرحوم راغب اصفهانی در مفردات می گوید: «توکل» اگر با حرف «علی» بیاید به معنای اعتما

In [40]:
## 12. Test top reranker candidates across 20 random questions

import random
import json
from collections import Counter
from tqdm.auto import tqdm

random.seed(123)

sample_records = random.sample(
    list(records_by_id.values()),
    20,
)

top_candidate_results = []

for record in tqdm(
    sample_records,
    desc="LLM labeling",
):
    candidate_meta = record["candidates"][0]

    candidate_id = candidate_meta["candidate_id"]
    candidate_text = records_by_id[candidate_id]["positive"]

    prompt = STRICT_PROMPT_TEMPLATE.format(
        question=record["query"],
        positive=record["positive"],
        candidate=candidate_text,
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
        max_tokens=20,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "candidate_label",
                "schema": {
                    "type": "object",
                    "properties": {
                        "label": {
                            "type": "string",
                            "enum": [
                                "negative",
                                "equivalent",
                                "uncertain",
                            ],
                        }
                    },
                    "required": ["label"],
                    "additionalProperties": False,
                },
            },
        },
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        },
    )

    result = json.loads(
        response.choices[0].message.content
    )

    top_candidate_results.append(
        {
            "query_id": record["id"],
            "candidate_id": candidate_id,
            "reranker_score": candidate_meta["reranker_score"],
            "label": result["label"],
        }
    )

print("\nLabel counts:")
print(Counter(x["label"] for x in top_candidate_results))

LLM labeling: 100%|██████████| 20/20 [00:06<00:00,  2.89it/s]


Label counts:
Counter({'negative': 12, 'equivalent': 8})


In [41]:
## 13. Run a 200-candidate production test

from collections import Counter
from tqdm.auto import tqdm
import json

TEST_OUTPUT_PATH = Path(
    "../data/processed/porseman_llm_labels_test.jsonl"
)

test_pairs = []

for record in records_by_id.values():
    for candidate_meta in record["candidates"]:
        test_pairs.append(
            (record, candidate_meta)
        )

        if len(test_pairs) == 200:
            break

    if len(test_pairs) == 200:
        break


label_counts = Counter()

with TEST_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    for record, candidate_meta in tqdm(
        test_pairs,
        desc="LLM labeling",
    ):
        candidate_id = candidate_meta["candidate_id"]
        candidate_text = records_by_id[candidate_id]["positive"]

        prompt = STRICT_PROMPT_TEMPLATE.format(
            question=record["query"],
            positive=record["positive"],
            candidate=candidate_text,
        )

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            temperature=0,
            max_tokens=20,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "candidate_label",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "label": {
                                "type": "string",
                                "enum": [
                                    "negative",
                                    "equivalent",
                                    "uncertain",
                                ],
                            }
                        },
                        "required": ["label"],
                        "additionalProperties": False,
                    },
                },
            },
            extra_body={
                "chat_template_kwargs": {
                    "enable_thinking": False
                }
            },
        )

        result = json.loads(
            response.choices[0].message.content
        )

        label = result["label"]
        label_counts[label] += 1

        output_record = {
            "query_id": record["id"],
            "candidate_id": candidate_id,
            "label": label,
        }

        f.write(
            json.dumps(
                output_record,
                ensure_ascii=False,
            )
            + "\n"
        )


print("\nLabel counts:")
print(label_counts)

print(f"\nSaved: {TEST_OUTPUT_PATH}")


LLM labeling: 100%|██████████| 200/200 [01:19<00:00,  2.53it/s]


Label counts:
Counter({'negative': 182, 'equivalent': 18})

Saved: ..\data\processed\porseman_llm_labels_test.jsonl


In [42]:
## 14. Parallel + resumable LLM labeling test

import json
import time

from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm


LLM_LABELS_PATH = Path(
    "../data/processed/porseman_llm_labels.jsonl"
)

MAX_WORKERS = 16
TEST_LIMIT = 1000
MAX_RETRIES = 3

JSON_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "candidate_label",
        "schema": {
            "type": "object",
            "properties": {
                "label": {
                    "type": "string",
                    "enum": [
                        "negative",
                        "equivalent",
                        "uncertain",
                    ],
                }
            },
            "required": ["label"],
            "additionalProperties": False,
        },
    },
}


# Load already completed pairs for resume
completed = set()

if LLM_LABELS_PATH.exists():
    with LLM_LABELS_PATH.open(
        encoding="utf-8"
    ) as f:
        for line in f:
            try:
                item = json.loads(line)
                completed.add(
                    (
                        item["query_id"],
                        item["candidate_id"],
                    )
                )
            except json.JSONDecodeError:
                pass


# Build pending pairs
pending = []

for record in records_by_id.values():
    for candidate in record["candidates"]:
        key = (
            record["id"],
            candidate["candidate_id"],
        )

        if key not in completed:
            pending.append(key)

        if len(pending) >= TEST_LIMIT:
            break

    if len(pending) >= TEST_LIMIT:
        break


print(f"Already completed: {len(completed):,}")
print(f"Pending in this run: {len(pending):,}")


def label_candidate(query_id, candidate_id):
    record = records_by_id[query_id]
    candidate_text = records_by_id[candidate_id]["positive"]

    prompt = STRICT_PROMPT_TEMPLATE.format(
        question=record["query"],
        positive=record["positive"],
        candidate=candidate_text,
    )

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                temperature=0,
                max_tokens=20,
                response_format=JSON_SCHEMA,
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    }
                },
            )

            result = json.loads(
                response.choices[0].message.content
            )

            return {
                "query_id": query_id,
                "candidate_id": candidate_id,
                "label": result["label"],
            }

        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise

            time.sleep(1)


label_counts = Counter()
errors = []


with LLM_LABELS_PATH.open(
    "a",
    encoding="utf-8",
) as output_file:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                label_candidate,
                query_id,
                candidate_id,
            ): (query_id, candidate_id)
            for query_id, candidate_id in pending
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="LLM labeling",
        ):
            query_id, candidate_id = futures[future]

            try:
                result = future.result()

                output_file.write(
                    json.dumps(
                        result,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

                # Save immediately for resume safety
                output_file.flush()

                label_counts[result["label"]] += 1

            except Exception as e:
                errors.append(
                    {
                        "query_id": query_id,
                        "candidate_id": candidate_id,
                        "error": str(e),
                    }
                )


print("\nLabels this run:")
print(label_counts)

print(f"\nErrors: {len(errors)}")
print(f"Total saved: {len(completed) + sum(label_counts.values()):,}")

Already completed: 0
Pending in this run: 1,000


LLM labeling: 100%|██████████| 1000/1000 [01:12<00:00, 13.78it/s]


Labels this run:
Counter({'negative': 897, 'equivalent': 103})

Errors: 0
Total saved: 1,000


In [47]:
## 15. Run full parallel + resumable LLM labeling

import json
import time

from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm


LLM_LABELS_PATH = Path(
    "../data/processed/porseman_llm_labels.jsonl"
)

MAX_WORKERS = 48
REQUEST_CHUNK_SIZE = 2000
MAX_RETRIES = 3


# Load completed pairs
completed = set()

if LLM_LABELS_PATH.exists():
    with LLM_LABELS_PATH.open(
        encoding="utf-8"
    ) as f:
        for line in f:
            try:
                item = json.loads(line)

                completed.add(
                    (
                        item["query_id"],
                        item["candidate_id"],
                    )
                )

            except json.JSONDecodeError:
                pass


# Build all remaining pairs
pending = []

for record in records_by_id.values():
    for candidate in record["candidates"]:
        key = (
            record["id"],
            candidate["candidate_id"],
        )

        if key not in completed:
            pending.append(key)


print(f"Already completed: {len(completed):,}")
print(f"Remaining: {len(pending):,}")


def label_candidate(query_id, candidate_id):
    record = records_by_id[query_id]
    candidate_text = records_by_id[candidate_id]["positive"]

    prompt = STRICT_PROMPT_TEMPLATE.format(
        question=record["query"],
        positive=record["positive"],
        candidate=candidate_text,
    )

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                temperature=0,
                max_tokens=20,
                response_format=JSON_SCHEMA,
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    }
                },
            )

            result = json.loads(
                response.choices[0].message.content
            )

            return {
                "query_id": query_id,
                "candidate_id": candidate_id,
                "label": result["label"],
            }

        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise

            time.sleep(1)


label_counts = Counter()
errors = []


with LLM_LABELS_PATH.open(
    "a",
    encoding="utf-8",
) as output_file:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        with tqdm(
            total=len(pending),
            desc="LLM labeling",
        ) as progress:

            for chunk_start in range(
                0,
                len(pending),
                REQUEST_CHUNK_SIZE,
            ):
                chunk = pending[
                    chunk_start:
                    chunk_start + REQUEST_CHUNK_SIZE
                ]

                futures = {
                    executor.submit(
                        label_candidate,
                        query_id,
                        candidate_id,
                    ): (query_id, candidate_id)
                    for query_id, candidate_id in chunk
                }

                for future in as_completed(futures):
                    query_id, candidate_id = futures[future]

                    try:
                        result = future.result()

                        output_file.write(
                            json.dumps(
                                result,
                                ensure_ascii=False,
                            )
                            + "\n"
                        )

                        output_file.flush()

                        label_counts[result["label"]] += 1

                    except Exception as e:
                        errors.append(
                            {
                                "query_id": query_id,
                                "candidate_id": candidate_id,
                                "error": str(e),
                            }
                        )

                    progress.update(1)


print("\nLabels this run:")
print(label_counts)

print(f"\nErrors: {len(errors):,}")
print(
    f"Total completed: "
    f"{len(completed) + sum(label_counts.values()):,}"
)

Already completed: 255,900
Remaining: 240


LLM labeling: 100%|██████████| 240/240 [00:12<00:00, 19.27it/s]


Labels this run:
Counter({'negative': 220, 'equivalent': 20})

Errors: 0
Total completed: 256,140


In [48]:
## 17. Final validation of LLM labels

VALID_LABELS = {
    "negative",
    "equivalent",
    "uncertain",
}

seen_pairs = set()
duplicate_pairs = 0
invalid_labels = 0
total_rows = 0

with LLM_LABELS_PATH.open(
    encoding="utf-8"
) as f:
    for line in f:
        item = json.loads(line)
        total_rows += 1

        key = (
            item["query_id"],
            item["candidate_id"],
        )

        if key in seen_pairs:
            duplicate_pairs += 1

        seen_pairs.add(key)

        if item["label"] not in VALID_LABELS:
            invalid_labels += 1


print(f"Rows in label file: {total_rows:,}")
print(f"Unique pairs: {len(seen_pairs):,}")
print(f"Duplicate pairs: {duplicate_pairs:,}")
print(f"Invalid labels: {invalid_labels:,}")

Rows in label file: 256,140
Unique pairs: 256,140
Duplicate pairs: 0
Invalid labels: 0


In [49]:
## 18. Count valid negatives per query

from collections import Counter

labels_by_pair = {}

with LLM_LABELS_PATH.open(
    encoding="utf-8"
) as f:
    for line in f:
        item = json.loads(line)

        labels_by_pair[
            (
                item["query_id"],
                item["candidate_id"],
            )
        ] = item["label"]


negative_counts = []

for record in records_by_id.values():
    count = 0

    for candidate in record["candidates"]:
        label = labels_by_pair[
            (
                record["id"],
                candidate["candidate_id"],
            )
        ]

        if label == "negative":
            count += 1

    negative_counts.append(count)


distribution = Counter(negative_counts)

print(f"Queries: {len(negative_counts):,}")
print(f"Minimum negatives: {min(negative_counts)}")
print(f"Median negatives: {sorted(negative_counts)[len(negative_counts)//2]}")
print(f"Queries with >= 7 negatives: {sum(x >= 7 for x in negative_counts):,}")
print(f"Queries with < 7 negatives: {sum(x < 7 for x in negative_counts):,}")

print("\nNegative-count distribution:")
for count in sorted(distribution):
    print(f"{count:2d} negatives: {distribution[count]:,} queries")

Queries: 12,807
Minimum negatives: 0
Median negatives: 19
Queries with >= 7 negatives: 12,752
Queries with < 7 negatives: 55

Negative-count distribution:
 0 negatives: 1 queries
 2 negatives: 4 queries
 3 negatives: 5 queries
 4 negatives: 13 queries
 5 negatives: 16 queries
 6 negatives: 16 queries
 7 negatives: 33 queries
 8 negatives: 43 queries
 9 negatives: 59 queries
10 negatives: 91 queries
11 negatives: 97 queries
12 negatives: 153 queries
13 negatives: 221 queries
14 negatives: 382 queries
15 negatives: 532 queries
16 negatives: 769 queries
17 negatives: 1,042 queries
18 negatives: 1,589 queries
19 negatives: 2,742 queries
20 negatives: 4,999 queries


In [50]:
## 19. Build training dataset with top-7 LLM-reviewed hard negatives

NEGATIVES_PER_QUERY = 7

training_records = []
skipped_query_ids = []

for record in records_by_id.values():
    valid_negatives = []

    for candidate in record["candidates"]:
        candidate_id = candidate["candidate_id"]

        label = labels_by_pair[
            (
                record["id"],
                candidate_id,
            )
        ]

        if label == "negative":
            valid_negatives.append(candidate)

    if len(valid_negatives) < NEGATIVES_PER_QUERY:
        skipped_query_ids.append(record["id"])
        continue

    valid_negatives = sorted(
        valid_negatives,
        key=lambda x: x["reranker_score"],
        reverse=True,
    )

    selected = valid_negatives[:NEGATIVES_PER_QUERY]

    training_records.append(
        {
            "id": record["id"],
            "query": record["query"],
            "positive": record["positive"],
            "negatives": [
                records_by_id[item["candidate_id"]]["positive"]
                for item in selected
            ],
            "negative_ids": [
                item["candidate_id"]
                for item in selected
            ],
            "retrieval_rank": [
                item["retrieval_rank"]
                for item in selected
            ],
            "reranker_score": [
                item["reranker_score"]
                for item in selected
            ],
        }
    )


print(f"Training records: {len(training_records):,}")
print(f"Skipped queries: {len(skipped_query_ids):,}")
print(f"Negatives per record: {len(training_records[0]['negatives'])}")

Training records: 12,752
Skipped queries: 55
Negatives per record: 7


In [51]:
## 20. Inspect one final training record

sample = training_records[0]

print("ID:")
print(sample["id"])

print("\nQUESTION:")
print(sample["query"])

print("\nPOSITIVE:")
print(sample["positive"])

print("\nSELECTED HARD NEGATIVES:")

for i, (
    negative_id,
    negative_text,
    retrieval_rank,
    reranker_score,
) in enumerate(
    zip(
        sample["negative_ids"],
        sample["negatives"],
        sample["retrieval_rank"],
        sample["reranker_score"],
    ),
    start=1,
):
    print("\n" + "-" * 100)
    print(f"Negative #{i}")
    print(f"ID: {negative_id}")
    print(f"Retrieval rank: {retrieval_rank}")
    print(f"Reranker score: {reranker_score:.4f}")
    print(negative_text[:500])

ID:
porseman-12457

QUESTION:
روزه‌هایى که در اوایل سن تکلیف به جا نیاورده‌ام، علاوه بر قضا کفّاره هم دارد؟

POSITIVE:
همه مراجع: هر مقدار از روزه‌ها را که نگرفته‌اید، باید قضا کنید و افزون بر آن، براى هر روز نیز باید کفّاره بدهید؛ یعنى، دو ماه روزه بگیرید یا شصت فقیر را سیر کنید و یا به هر کدام یک مد (تقریبا ده سیر) طعام (گندم یا جو و مانند آن) به آنها بدهید. [ توضیح‌المسائل مراجع، م 1660؛ وحید، توضیح‌المسائل، م 1668.]

SELECTED HARD NEGATIVES:

----------------------------------------------------------------------------------------------------
Negative #1
ID: porseman-12610
Retrieval rank: 1
Reranker score: 0.8966
همه مراجع: خیر، تنها قضاى روزه‌ها واجب است و کفّاره ندارد؛ ولى اگر قضاى روزه‌ها را تا ماه رمضان سال بعد به تأخیر اندازد، به جهت تأخیر، باید براى هر روز، یک مد طعام کفّاره بدهد.[ توضیح‌المسائل مراجع، م 1705؛ وحید، توضیح‌المسائل، م1713.]

----------------------------------------------------------------------------------------------------
Negative #2
ID: porseman-35427
Retriev

In [52]:
## 21. Export final embedding training dataset

import json

FINAL_OUTPUT_PATH = Path(
    "../data/processed/porseman_embedding_train.jsonl"
)

with FINAL_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    for record in training_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"Saved: {FINAL_OUTPUT_PATH}")
print(f"Records: {len(training_records):,}")
print(
    f"Size: "
    f"{FINAL_OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB"
)

Saved: ..\data\processed\porseman_embedding_train.jsonl
Records: 12,752
Size: 249.25 MB


In [53]:
## 22. Export minimal embedding training dataset

MINIMAL_OUTPUT_PATH = Path(
    "../data/processed/porseman_embedding_train_minimal.jsonl"
)

with MINIMAL_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    for record in training_records:
        minimal_record = {
            "query": record["query"],
            "positive": record["positive"],
            "negatives": record["negatives"],
        }

        f.write(
            json.dumps(
                minimal_record,
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"Saved: {MINIMAL_OUTPUT_PATH}")
print(f"Records: {len(training_records):,}")
print(
    f"Size: "
    f"{MINIMAL_OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB"
)

Saved: ..\data\processed\porseman_embedding_train_minimal.jsonl
Records: 12,752
Size: 244.70 MB
